In [5]:
import numpy as np
import cv2
import joblib
import os
import glob
import pandas as pd
import tifffile as tiff
from tqdm import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

print("Imports successful!")

Imports successful!


In [2]:
# --- Configuration ---

# 1. Path to your RAW data
DATA_PATH = '../data/raw/HyperLeaf2024/HyperLeaf2024/' # <-- From your notebook 1
IMG_FOLDER = os.path.join(DATA_PATH, "images")
TRAIN_CSV_PATH = os.path.join(DATA_PATH, "train.csv")

# 2. Path for the FINAL pipeline object
MODEL_DIR = '../models/'
PIPELINE_PATH = os.path.join(MODEL_DIR, 'hyperleaf_full_pipeline.joblib')

# 3. Model's expected shape
MODEL_HEIGHT = 128
MODEL_WIDTH = 128
# --- !! NEW: Define sample size !! ---
NUM_SAMPLE_IMAGES = 50# We will use 50 images to fit the pipeline

In [3]:
# Ensure model directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

# Load the training CSV
try:
    train_labels_df = pd.read_csv(TRAIN_CSV_PATH)
    print(f"Loaded {len(train_labels_df)} records from train.csv")
except FileNotFoundError:
    print(f"ERROR: Cannot find train.csv at: {TRAIN_CSV_PATH}")
    raise # Stop the notebook

# --- !! NEW: Shuffle and sample the data !! ---
if len(train_labels_df) > NUM_SAMPLE_IMAGES:
    print(f"Sampling {NUM_SAMPLE_IMAGES} random images to build pipeline...")
    sample_df = train_labels_df.sample(n=NUM_SAMPLE_IMAGES, random_state=42)
else:
    print("Warning: Sample size is larger than dataset. Using all images.")
    sample_df = train_labels_df
# ---

all_training_pixels = []

print(f"Starting to load, resize, and stack pixels from {len(sample_df)} sample images...")
print("This will still take time, but much less RAM.")

# Loop over the SAMPLED dataframe
for index, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    img_id_int = int(row['ImageId'])
    img_id_str = str(img_id_int).zfill(5)
    img_path = os.path.join(IMG_FOLDER, f"{img_id_str}.tiff")

    try:
        # 1. Load the raw (48, 352, 204) image
        raw_image = tiff.imread(img_path)
        
        # 2. Apply spatial resizing
        resized_image = cv2.resize(
            raw_image, 
            (MODEL_WIDTH, MODEL_HEIGHT), 
            interpolation=cv2.INTER_LINEAR
        )
        # Shape is now (128, 128, 204)
        
        # 3. Reshape for scikit-learn
        num_bands = resized_image.shape[-1]
        reshaped_pixels = resized_image.reshape(-1, num_bands)
        
        all_training_pixels.append(reshaped_pixels)

    except Exception as e:
        print(f"Warning: Skipping {img_path}. Error: {e}")

# 4. Stack into one 2D array
if all_training_pixels:
    # This array will be much smaller now
    X_train_sample = np.vstack(all_training_pixels)
    print(f"\nFinished stacking data. Final array shape: {X_train_sample.shape}")
    print(f"(This is {X_train_sample.nbytes / 1e9:.2f} GB)")
else:
    print("ERROR: No data was loaded. Check your paths.")

Loaded 1590 records from train.csv
Sampling 50 random images to build pipeline...
Starting to load, resize, and stack pixels from 50 sample images...
This will still take time, but much less RAM.


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:03<00:00, 14.77it/s]



Finished stacking data. Final array shape: (819200, 352)
(This is 0.58 GB)


In [6]:
# Check for the new variable name 'X_train_sample'
if 'X_train_sample' in locals():
    print("Defining the full preprocessing pipeline...")

    full_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=3)),
        ('minmax', MinMaxScaler()) 
    ])

    print("Fitting the pipeline on the *sampled* training pixels...")
    # This fits the pipeline on the smaller, sampled array
    full_pipeline.fit(X_train_sample)
    
    # Save the fitted pipeline
    joblib.dump(full_pipeline, PIPELINE_PATH)
    
    print("\n--- Pipeline Build Complete! ---")
    print(f"Saved pipeline to: {PIPELINE_PATH}")
else:
    print("Skipping pipeline fit (no data).")

Defining the full preprocessing pipeline...
Fitting the pipeline on the *sampled* training pixels...

--- Pipeline Build Complete! ---
Saved pipeline to: ../models/hyperleaf_full_pipeline.joblib
